In [2]:
import gc
gc.collect()

554

In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [4]:
ruta_archivo_aprender = "../data/processed/final.csv"
ruta_archivo_provincias = "./shapefiles/provincia/provinciaPolygon.shp"

In [5]:
df = pd.read_csv(ruta_archivo_aprender, sep=',', decimal=".", low_memory=False)

In [11]:
print(sorted(df.columns))

['Nivel_Ed_Madre_Primaria_completo', 'Nivel_Ed_Madre_Secundaria_completo', 'Nivel_Ed_Madre_Terciario_universitario_posgrado_completo', 'Nivel_Ed_Padre_Primaria_completo', 'Nivel_Ed_Padre_Secundaria_completo', 'Nivel_Ed_Padre_Terciario_universitario_posgrado_completo', 'Nro_Encuestados', 'ambito_binario', 'ap02_2004', 'ap02_2005', 'ap02_2006', 'ap03_Femenino', 'ap03_Masculino', 'ap11_1', 'ap11_2', 'ap11_3', 'ap11_4', 'ap11_5', 'ap11_6', 'ap12_2', 'ap12_3', 'ap12_4', 'ap12_5', 'ap12_6', 'ap12_7', 'ap12_Vivo_solo', 'ap14_No', 'ap14_Sí', 'ap23_Blanco', 'ap23_Sí_recibí_dinero_por_todas_ellas', 'ap24_No_no_fui_al_jardín', 'ap24_Sí_desde_sala_de_3_o_antes', 'ap24_Sí_desde_sala_de_4', 'ap24_Sí_desde_sala_de_5', 'ap25a_No', 'ap25a_Sí_dos_o_más_veces', 'ap25a_Sí_una_vez', 'ap27_De_5_a_14_faltas', 'ap27_Menos_de_5_faltas', 'lat', 'ldesemp_Avanzado', 'ldesemp_Básico', 'ldesemp_Por_debajo_del_nivel_básico', 'ldesemp_Satisfactorio', 'lon', 'mdesemp_Avanzado', 'mdesemp_Básico', 'mdesemp_Por_debajo_de

In [6]:
geometry = [
    Point(xy) for xy in zip(df['lon'], df['lat'])
]
gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

In [7]:
provincias = gpd.read_file(ruta_archivo_provincias)

In [12]:
print(provincias.columns)

Index(['gid', 'entidad', 'fna', 'gna', 'nam', 'in1', 'fdc', 'sag', 'geometry'], dtype='object')


In [13]:
print(provincias['nam'].unique())

['Ciudad Autónoma de Buenos Aires' 'San Luis' 'Santa Fe' 'Tucumán' 'Jujuy'
 'Santiago del Estero' 'Córdoba' 'La Pampa' 'Chaco' 'Formosa' 'Chubut'
 'Río Negro' 'Misiones' 'Salta' 'Buenos Aires'
 'Tierra del Fuego, Antártida e Islas del Atlántico Sur' 'Entre Ríos'
 'Corrientes' 'Mendoza' 'San Juan' 'Neuquén' 'La Rioja' 'Catamarca'
 'Santa Cruz']


In [8]:
resultado = gpd.sjoin(
    gdf,
    provincias,
    how="left",
    predicate="within"
)

print(resultado[['lat', 'lon', 'nam']])

            lat        lon                                                nam
0    -34.609558 -58.388790                    Ciudad Autónoma de Buenos Aires
1    -34.609558 -58.388790                    Ciudad Autónoma de Buenos Aires
2    -34.627458 -58.503098                    Ciudad Autónoma de Buenos Aires
3    -34.627458 -58.503098                    Ciudad Autónoma de Buenos Aires
4    -34.603006 -58.500717                    Ciudad Autónoma de Buenos Aires
...         ...        ...                                                ...
5917 -53.785828 -67.701856  Tierra del Fuego, Antártida e Islas del Atlánt...
5918 -53.785828 -67.701856  Tierra del Fuego, Antártida e Islas del Atlánt...
5919 -54.511349 -67.195400  Tierra del Fuego, Antártida e Islas del Atlánt...
5920 -54.807306 -68.308413  Tierra del Fuego, Antártida e Islas del Atlánt...
5921 -54.807306 -68.308413  Tierra del Fuego, Antártida e Islas del Atlánt...

[5922 rows x 3 columns]


In [12]:
prom = (
    resultado
    .groupby("nam")["Nro_Encuestados"]
    .mean()
    .reset_index()
)

In [15]:
resultado.groupby("nam").size()


nam
Buenos Aires                                             1447
Catamarca                                                 133
Chaco                                                     263
Chubut                                                     94
Ciudad Autónoma de Buenos Aires                           326
Corrientes                                                263
Córdoba                                                   350
Entre Ríos                                                273
Formosa                                                   114
Jujuy                                                     173
La Pampa                                                  161
La Rioja                                                  124
Mendoza                                                   231
Misiones                                                  228
Neuquén                                                   113
Río Negro                                                 127
Salt

In [18]:
conteo = (
    resultado
    .groupby("nam")
    .size()
    .reset_index(name="cantidad")
)


In [19]:
mapa = provincias.merge(
    conteo,
    on="nam",
    how="left"
)

In [13]:
mapa = provincias.merge(
    prom,
    on="nam",
    how="left"
)

In [14]:
geojson = mapa.geometry.__geo_interface__

In [ ]:
import plotly.express as px

fig = px.choropleth_map(
    mapa,
    geojson=geojson,
    locations=mapa.index,
    color="Nro_Encuestados",
    hover_name="nam",
    map_style="carto-positron",
    center={"lat": -38, "lon": -64},
    zoom=3.2,
    opacity=0.7
)

fig.show()